In [1]:
from tempfile import TemporaryDirectory

# imports
import numpy as np
import numpy.linalg as LA

from utils.bench import Benchmark
from base.qda import QDA, TensorizedQDA, FasterQDA, EfficientQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.datasets import (get_iris_dataset, get_letters_dataset,
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)
from utils.datasets import split_transpose

In [2]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()
y_full_encoded = label_encode(y_full)

In [3]:
FQDA = FasterQDA()

test_sz = 0.3
rng = 6663

# Generate current train-test split
X_train, X_test, y_train, y_test = split_transpose(
    X_full, y_full_encoded,
    test_size=test_sz,
    random_state=rng
)
FQDA.fit(X_train, y_train)

In [4]:
fqda_predict = FQDA.predict(X_test)
print(fqda_predict)

[[0 0 1 1 1 2 0 0 2 1 2 0 0 2 2 1 0 0 1 0 0 0 1 1 1 1 1 1 1 1 0 1 1 2 1 1
  1 0 2 0 1 1 0 0 0 1 0 0 0 1 1 0 1 0]]


In [5]:
# Punto 5

inv_covs = FQDA.inv_covs[0]
print(f"Shape matriz de covarianzas: {inv_covs.shape}")
# 13x13
mean = FQDA.means[0]
print(f"Shape de matriz de medias: {mean.shape}")
# 13x1

X_c = X_test - mean
print(f"Shape de matriz de Xc: {X_c.shape}")
# 13x54

A = X_c.T @ inv_covs
print(f"Shape de A: {A.shape}")
# 54x13
B = X_c   
print(f"Shape de B: {B.shape}\n")
# 13x54

diagonal_lenta = np.diagonal(A @ B)

# Podemos trasponer A o B, da igual al final
diagonal_rapida0 = np.sum(A.T*B,axis=0)
diagonal_rapida1 = np.sum(A*B.T,axis=1)

# Usamos allclose porque al sumar en distinto orden puede diferir el redondeo y que con array_equal no coincidan

print("Mismo resultado?")
print(np.allclose(diagonal_lenta, diagonal_rapida0))
print(np.allclose(diagonal_lenta, diagonal_rapida1))


Shape matriz de covarianzas: (13, 13)
Shape de matriz de medias: (13, 1)
Shape de matriz de Xc: (13, 54)
Shape de A: (54, 13)
Shape de B: (13, 54)

Mismo resultado?
True
True


In [6]:
# Punto 6
EQDA = EfficientQDA()

EQDA.fit(X_train, y_train)
eqda_predict = EQDA.predict(X_test)

print(f"Mismo resultado? {np.array_equal(fqda_predict, eqda_predict)}\n")

# Comparación de proceso FQDA y EQDA
centered_X = X_test - FQDA.tensor_means

cross_products = centered_X.transpose(0, 2, 1) @ FQDA.tensor_inv_cov @ centered_X
print("FasterQDA")
print(f"Shape matriz de producto: {cross_products.shape}\n")

print("EfficientQDA")
A = centered_X.transpose(0, 2, 1) @ FQDA.tensor_inv_cov
B = centered_X
print(f"Shape matriz intermedia: {(A.transpose(0, 2, 1) * B).shape}")


Mismo resultado? True

FasterQDA
Shape matriz de producto: (3, 54, 54)

EfficientQDA
Shape matriz intermedia: (3, 13, 54)


In [7]:
# Punto 7
benchmark = Benchmark(X_full, y_full_encoded)

benchmark.bench(QDA)
benchmark.bench(TensorizedQDA)
benchmark.bench(FasterQDA)
benchmark.bench(EfficientQDA)

benchmark.summary(baseline="QDA")

Benching params:
Total runs: 1200
Warmup runs: 100
Peak Memory usage runs: 100
Running time runs: 1000
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.086292,0.062051,0.969354,0.276107,0.984593,0.018173,0.000851,0.007697,0.000422,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.089625,0.030291,0.448208,0.055415,0.984593,0.018082,0.000785,0.011986,0.000140,0.962811,2.162732,1.005063,0.642187
FasterQDA,0.085833,1.142081,0.023958,0.071040,0.984593,0.018082,0.000801,0.109116,0.000204,1.005347,40.460487,1.005063,0.070541
EfficientQDA,0.086708,0.033505,0.022000,0.011853,0.984593,0.018082,0.000804,0.060089,0.000215,0.995202,44.061442,1.005063,0.128095


In [8]:
# Para comprobar mejor, uso un dataset más grande
X_letters, y_letters = get_letters_dataset()
y_letters_encoded = label_encode(y_letters)

test_sz = 0.3
rng = 6663

# Generate current train-test split
X_train, X_test, y_train, y_test = split_transpose(
    X_letters, y_letters_encoded,
    test_size=test_sz,
    random_state=rng
)

n, p = X_letters.shape
k = len(np.unique(y_letters_encoded))
print(f"Observaciones: {n}, features: {p}, clases: {k}")

benchmark = Benchmark(X_letters, y_letters_encoded)

benchmark.bench(QDA)
benchmark.bench(TensorizedQDA)
benchmark.bench(FasterQDA)
benchmark.bench(EfficientQDA)

benchmark.summary(baseline="QDA")

Observaciones: 20000, features: 16, clases: 26
Benching params:
Total runs: 1200
Warmup runs: 100
Peak Memory usage runs: 100
Running time runs: 1000
Train size rows (approx): 14000
Test size rows (approx): 6000
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,3.103166,7.586256,913.441020,349.062058,0.884581,0.249794,0.002487,0.112493,0.000274,1.000000,1.000000,1.0,1.000000
TensorizedQDA,5.214146,8.423809,350.359876,24.731570,0.884581,0.249794,0.002492,0.169403,0.000108,0.595144,2.607151,1.0,0.664053
FasterQDA,6.225167,8.794738,1187.117625,499.709315,0.884581,0.249794,0.002490,7179.315666,0.000594,0.498487,0.769461,1.0,0.000016
EfficientQDA,3.786646,2.399063,8.391271,2.359606,0.884581,0.249794,0.002490,58.435951,0.000027,0.819503,108.856105,1.0,0.001925


## 4. ¿Dónde aparece la matriz de `n × n`?

En `FasterQDA.predict`, la línea que calcula `cross_products` es la que construye esa matriz:

```python
cross_products = centered_X.transpose(0, 2, 1) @ self.tensor_inv_cov @ centered_X
```

Si hay `k` clases, `p` features y `n` observaciones, `centered_X` tiene shape `(k, p, n)` y `self.tensor_inv_cov` tiene `(k, p, p)`. El producto tiene entonces shape `(k, n, n)`: **para cada clase se genera una matriz de `n × n`** con las interacciones entre observaciones. En este ejemplo, con 3 clases y 54 observaciones de prueba, `cross_products` tiene shape `(3, 54, 54)`.

`np.diagonal(cross_products, axis1=-2, axis2=-1)` extrae los `n` términos necesarios para cada clase y devuelve un array de shape `(k, n)`, pero la matriz completa ya fue calculada.

## 5. Demostración

Dado $A \in \mathbb{R}^{n \times p}$ y $B \in \mathbb{R}^{p \times n}$, queremos demostrar que:

$$
\text{diag}(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis = 1)
$$

**Lado izquierdo:** 

Por definición, el elemento $(i, j)$ de $AB$ (producto de matrices) es:

$$
(AB)_{ij} = \sum_{m=1}^{p} A_{im} \, B_{mj}
$$

La diagonal está formada solo por los elementos donde $j = i$, por lo que para cada $i = 1,..., n$:

$$
\text{diag}(AB)_i = (AB)_{ii} = \sum_{m=1}^{p} A_{im} \, B_{mi}
$$

**Lado derecho:** 

La traspuesta de una matriz se define como $(B^T)_{im} = B_{mi}$.

Como $B^T \in \mathbb{R}^{n \times p}$, tiene el mismo shape que $A$, el producto elemento a elemento queda:

$$
(A \odot B^T)_{im} = A_{im} \, (B^T)_{im} = A_{im} \, B_{mi}
$$

Sumando sobre las columnas de esa matriz (sobre axis = 1), para cada fila $i$ se obtiene:

$$
\left( \sum_{cols} A \odot B^T \right)_i = \sum_{k=1}^{p} A_{im} \, B_{mi}
$$

Por lo tanto:

$$
\text{diag}(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis = 1) = \sum_{m=1}^{p} A_{im} \, B_{mi}
$$

Nótese que el lado izquiero requiere construir la matriz $AB \in \mathbb{R}^{n \times n}$ con $n^2$ elementos pero solo términamos utilizando la diagonal de $n$ elementos, mientras que el lado derecho usa una matrix de $n \times p$. Generalmente al trabajar con métodos de clasificación, $p \ll n$ porque vamos a tener muchas más observaciones que features.

## 7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

A simple vista, los cuatro modelos tienen la misma exactitud promedio, lo cuál tiene sentido ya que llegan a un mismo resultado. Optimizar el modelo entones no afecta su exactitud.
También comparten tiempos de entrenamiento y uso de memoria para entrenamiento muy similares. Esto se debe a que utilizan un mismo `fit` (heredado de la clase `QDA`).
La gran diferencia se nota en el testeo: `QDA` destaca por ser el método más lento de testeo mientras que `FasterQDA` y `EfficientQDA` son los métodos más veloces por lejos.
`TensorizedQDA` también denota una mejora substancial (pero mucho más leve) en comparación con `QDA`. Eliminar el `for` de clases (k=3) ayuda, pero `TensorizedQDA` sigue haciendo un `for` para recorrer las 53 observaciones.
Hay un salto colosal de velocidad entre `TensorizedQDA` y `FasterQDA`/`EfficientQDA`. Aunque `FasterQDA`/`EfficientQDA` técnicamente hace más cuentas (para cada clase genera una matriz de nxn siendo n la cantidad de observaciones), resultan mucho más rápidas que tener que recorrer el ciclo `for`. Si n fuera mayor, esta diferencia sería incluso más notoria. Esto demuestra también lo costosos que son los `for` en Python.
`EfficientQDA` es práctimante igual de rápida `FasterQDA` (personalmente, esperaba una mayor diferencia incluso). Esto puede deberse a que no contamos con suficientes observaciones para que se note mejor la ventaja de `EfficientQDA`.
Por otro lado, `QDA` es el modelo que menos memoria utiliza durante el testeo, mientras que `FasterQDA` es el que más utiliza. Es un trade-off lógico ya que vectorizar es más rápido para realizar los cálculos pero significa que tenemos que tener todo en memoria de una. Como `QDA` tiene sus ciclos for, toma una observación y clase a la vez así que no ocupa tanto espacio.
Cumpliendo con su nombre, `EfficientQDA` usa casi la mitad de memoria que `FasterQDA` sin sacrificar la velocidad. Como se notó en el punto anterior, el shape de la matriz de producto es de (k,n,n) (o sea, (3,54,54)) mientras que la de `EfficientQDA` es de (k,p,n) (o sea, (3,13,54)). Por esta razón, el modelo ocupa menos espacio.
Los resultados son más o menos lo que se esperaba, con la excepción de que se esperaba que `EfficientQDA` fuera bastante más rápido que `FasterQDA`. Pero al tener un n relativamente pequeño, la diferencia no es tan notoria.
Para corroborar el efecto de n sobre `FasterQDA`, se volvió a correr el benchmarking con un dataset mucho más extenso. En este caso quedo evidenciado de forma muy clara que que `EfficientQDA` es mucho más veloz que `FasterQDA` cuando el dataset tiene muchas muestras (test median de 8.39ms vs 1187.12ms). Incluso `FasterQDA` termino siendo más lento que `QDA`. Esto demuestra que el desempeño de `FasterQDA` esta ligado a la cantidad de muestras dada su matriz de producto (cuyo tamaño crece con $n^2$), mientras que `EfficientQDA` se ve menos afectada por este factor (al tener $n \cdot p$ elementos).